<a href="https://colab.research.google.com/github/vaishali27-c/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

Lane: Refresh / Content Opportunity Scoring

### Rule

Pages should receive a higher priority score when they have:

- High search volume
- High impressions
- Older content (staleness)
- Lower click-through rate

### Reason Code

STALE_CONTENT

### Action

Review for Content Refresh

This is a simple rule-based baseline that ranks pages for review. It does not use future information or label-derived features.

## Baseline Rule

Lane: Refresh / Content Opportunity Scoring

### Rule

Pages should receive a higher priority score when they have:

- High search volume
- High impressions
- Older content (staleness)
- Lower click-through rate

### Reason Code

STALE_CONTENT

### Action

Review for Content Refresh

This is a simple rule-based baseline that ranks pages for review. It does not use future information or label-derived features.

In [3]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [5]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HuggingFace,
    TOKEN '{HF_TOKEN}'
)
""")

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM {src}")

con.sql("SHOW TABLES").df()

,name
0,dim_clients
1,dim_content
2,fact_daily
3,fact_daily_sample
4,fact_query_90d


In [6]:
df = con.sql("""
SELECT *
FROM fact_daily_sample
LIMIT 10000
""").df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [8]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
import pandas as pd

ranked = df.copy()

# Baseline scoring rule
ranked["baseline_score"] = (
    ranked["gsc_impressions"].fillna(0) * 0.4 +
    ranked["gsc_clicks"].fillna(0) * 0.3 +
    ranked["ga4_pageviews"].fillna(0) * 0.2 -
    ranked["gsc_avg_position"].fillna(0) * 0.1
)

# Reason code
ranked["reason_code"] = "REFRESH_PRIORITY"

# Action label
ranked["action"] = "Review for Refresh"

# Rank pages
ranked = ranked.sort_values("baseline_score", ascending=False)

# Save CSV
import os
os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
top_20 = ranked.head(20).copy()
top_20['confidence_note'] = ''
top_20['what_would_make_it_wrong'] = ''

display(top_20[['content_hash_id', 'baseline_score', 'action', 'reason_code', 'confidence_note', 'what_would_make_it_wrong']])

,content_hash_id,baseline_score,action,reason_code,confidence_note,what_would_make_it_wrong
9011,content_f107e54b10b43725,2875.630213,Review for Refresh,REFRESH_PRIORITY,,
8756,content_88ff1c6680db0a45,1655.788242,Review for Refresh,REFRESH_PRIORITY,,
9032,content_c556c7369fb2fd06,1005.021956,Review for Refresh,REFRESH_PRIORITY,,
9131,content_9111c7d2691be9ad,995.179751,Review for Refresh,REFRESH_PRIORITY,,
8791,content_03621e012733c047,919.623174,Review for Refresh,REFRESH_PRIORITY,,
9021,content_7ff032d3024d35e4,762.544864,Review for Refresh,REFRESH_PRIORITY,,
8854,content_76c287232b54857b,735.845172,Review for Refresh,REFRESH_PRIORITY,,
9150,content_870b507d6cb3d481,646.598019,Review for Refresh,REFRESH_PRIORITY,,
8513,content_a603f13549019b16,616.855052,Review for Refresh,REFRESH_PRIORITY,,
8801,content_d1582b1c3ba7f221,577.308512,Review for Refresh,REFRESH_PRIORITY,,


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

For the 'Weak picks + leakage check', we need to manually review the `top_20` items and look for two main things:

1.  **Weak Picks**: Are there any content items in the top 20 that, based on your understanding of the business or content strategy, seem out of place or shouldn't be highly prioritized for refresh? If so, why do they seem 'weak'?
2.  **Leakage**: Does the `baseline_score` or the ranking itself seem to be influenced by information that wouldn't genuinely be available at the time of scoring, or by any product-specific flags that were not intended to be part of this generic baseline? This could indicate a 'leak' of future data or specific product knowledge into the general scoring logic.

Use the `confidence_note` and `what_would_make_it_wrong` columns below to record your observations for each of the top 20 items.

In [12]:
display(top_20[['content_hash_id', 'baseline_score', 'action', 'reason_code', 'confidence_note', 'what_would_make_it_wrong']])

,content_hash_id,baseline_score,action,reason_code,confidence_note,what_would_make_it_wrong
9011,content_f107e54b10b43725,2875.630213,Review for Refresh,REFRESH_PRIORITY,,
8756,content_88ff1c6680db0a45,1655.788242,Review for Refresh,REFRESH_PRIORITY,,
9032,content_c556c7369fb2fd06,1005.021956,Review for Refresh,REFRESH_PRIORITY,,
9131,content_9111c7d2691be9ad,995.179751,Review for Refresh,REFRESH_PRIORITY,,
8791,content_03621e012733c047,919.623174,Review for Refresh,REFRESH_PRIORITY,,
9021,content_7ff032d3024d35e4,762.544864,Review for Refresh,REFRESH_PRIORITY,,
8854,content_76c287232b54857b,735.845172,Review for Refresh,REFRESH_PRIORITY,,
9150,content_870b507d6cb3d481,646.598019,Review for Refresh,REFRESH_PRIORITY,,
8513,content_a603f13549019b16,616.855052,Review for Refresh,REFRESH_PRIORITY,,
8801,content_d1582b1c3ba7f221,577.308512,Review for Refresh,REFRESH_PRIORITY,,


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.